# Video Factory: Google Colab GPU Worker (Interactive & Debug)

This notebook provides interactive inspection, debugging, and Google Colab MCP connection.
The primary automated production pipeline runs headless via `google-colab-cli`.

### Capabilities:
1. GPU hardware check (`nvidia-smi`)
2. Lightweight dependency installation
3. `faster-whisper` CUDA library validation
4. Audio transcription with segment timestamps
5. JSON / SRT export

In [ ]:
# Step 1: Detect GPU hardware & CUDA environment
!nvidia-smi

In [ ]:
# Step 2: Install minimal required dependencies
!pip install --quiet faster-whisper==1.2.1 ctranslate2==4.6.0 nvidia-cublas-cu12 nvidia-cudnn-cu12

In [ ]:
# Step 3: Configure LD_LIBRARY_PATH for CUDA and test faster-whisper
import os, sys, importlib.util
from faster_whisper import WhisperModel

lib_dirs = []
for mod in ('nvidia.cublas.lib', 'nvidia.cudnn.lib'):
    spec = importlib.util.find_spec(mod)
    if spec and spec.submodule_search_locations:
        lib_dirs.extend(spec.submodule_search_locations)

os.environ['LD_LIBRARY_PATH'] = os.pathsep.join(lib_dirs + [os.environ.get('LD_LIBRARY_PATH', '')])
print('CUDA library paths configured:', lib_dirs)

# Load Whisper model on CUDA (T4 default)
model = WhisperModel('large-v3-turbo', device='cuda', compute_type='float16')
print('faster-whisper model loaded successfully on CUDA!')

In [ ]:
# Step 4: Transcribe sample audio and verify timestamped segments
import json

sample_audio = 'sample.wav'
# If testing without an uploaded file, create a dummy test file or transcribe target
if os.path.exists(sample_audio):
    segments, info = model.transcribe(sample_audio, beam_size=5, language='zh')
    results = []
    for s in segments:
        results.append({
            'start': round(float(s.start), 3),
            'end': round(float(s.end), 3),
            'text': s.text
        })
    print(f'Transcription complete: {len(results)} segments.')
    with open('transcript.json', 'w', encoding='utf-8') as f:
        json.dump({'text': ''.join(r['text'] for r in results), 'segments': results}, f, ensure_ascii=False, indent=2)
else:
    print(f'Upload an audio file to {sample_audio} to test transcription.')

In [ ]:
# Step 5: Export SRT and inspect output
if os.path.exists('transcript.json'):
    with open('transcript.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(json.dumps(data, ensure_ascii=False, indent=2)[:500])